In [1]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.append(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.append(os.path.join(os.getcwd(), "..", "..", "..", "..", ".."))
sys.path.append(os.path.join(os.getcwd(), "..", ".."))

In [2]:
import torch
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
import pandas as pd
from torch.utils.data import DataLoader
import torch.nn.functional as F
import statistics
import math
import numpy as np
from torch.nn.functional import normalize
from src.models.baseline.nlp.transformer.transformer import TransformerEncoder
from src.models.haven.haven import HAVEN
from src.utils import constants, nn_utils, utils, dataset_utils
from src.datasets.protein_sequence_with_id_dataset import ProteinSequenceDatasetWithID
from src.datasets.collations.padding_with_id import PaddingWithID

/home/blessyantony/miniconda3/envs/haven/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
input_file_path = os.path.join(os.getcwd(), "..","..", "..", "..", "input/data/coronaviridae/20240313/sarscov2/sarscov2_variants_spike_seqences_ncbivirus_20250106_who_variants_downsampled.csv")
input_df = pd.read_csv(input_file_path, index_col=0)
input_df

,accession_id,species,genbank_or_refseq,pangolin_lineage,seq_length,protein,virus_host_name,geo_location,seq
0,YP_009724390.1,surface glycoprotein [Severe acute respiratory...,RefSeq,B,1273,surface glycoprotein,homo sapiens,China,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
0,QUR41397.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,Egypt,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
1,UIX18801.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,USA: Illinois,MFVFLVLLPLVSSQCVXXTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
2,UFJ23379.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,USA: Florida,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3,UGZ97974.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,USA: California,MFVFLVLLPLVSXXCVNLTTRTXXXPAYTNSFTRGVYYPDKVFRSS...
...,...,...,...,...,...,...,...,...,...
95,UNN31516.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1270,surface glycoprotein,homo sapiens,USA: Florida,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
96,WAA85805.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1272,surface glycoprotein,homo sapiens,USA,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
97,UIQ58390.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1270,surface glycoprotein,homo sapiens,USA: South Carolina,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
98,WDU17786.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1270,surface glycoprotein,homo sapiens,Brazil,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...


In [4]:
input_df["pangolin_lineage"].unique()

array(['B', 'B.1.1.7', 'B.1.351', 'P.1', 'B.1.617.2', 'B.1.427',
       'B.1.429', 'P.2', 'B.1.525', 'P.3', 'B.1.526', 'B.1.617.1', 'C.37',
       'B.1.621', 'B.1.1.529', 'BA.1'], dtype=object)

In [5]:
who_designation_df = pd.read_csv(os.path.join(os.getcwd(), "..","..", "..", "..", "input/data/coronaviridae/20240313/sarscov2/sarscov2_who_designated_voc.csv"))
who_designation_df

,pango_lineage,who_variant,designation,first_designation_date,location
0,B,Wuhan-Hu-1,Index,12/01/2019,China
1,B.1.1.7,Alpha,VOC,12/08/2020,United Kingdom
2,B.1.351,Beta,VOC,12/18/2020,South Africa
3,P.1,Gamma,VOC,1/11/2021,Brazil
4,B.1.617.2,Delta,VOC,5/11/2021,India
5,B.1.427,Epsilon,VOI,3/5/2021,USA
6,B.1.429,Epsilon,VOI,3/5/2021,USA
7,P.2,Zeta,VOI,3/17/2021,Brazil
8,B.1.525,Eta,VOI,3/17/2021,Nigeria
9,P.3,Theta,VOI,3/24/2021,Philippines


In [6]:
who_designation_df["pango_who_annotation"] = who_designation_df["pango_lineage"] + " (" + who_designation_df["who_variant"] + ")" 
variant_order = who_designation_df["pango_who_annotation"].unique()
variant_order

array(['B (Wuhan-Hu-1)', 'B.1.1.7 (Alpha)', 'B.1.351 (Beta)',
       'P.1 (Gamma)', 'B.1.617.2 (Delta)', 'B.1.427 (Epsilon)',
       'B.1.429 (Epsilon)', 'P.2 (Zeta)', 'B.1.525 (Eta)', 'P.3 (Theta)',
       'B.1.526 (Iota)', 'B.1.617.1 (Kappa)', 'C.37 (Lambda)',
       'B.1.621 (Mu)', 'B.1.1.529 (Omicron)', 'BA.1 (Omicron)'],
      dtype=object)

In [7]:
who_designation_df.rename(columns={"pango_lineage": "pangolin_lineage"}, inplace=True)

In [8]:
input_df = pd.merge(input_df, who_designation_df, how="inner", on="pangolin_lineage")
input_df

,accession_id,species,genbank_or_refseq,pangolin_lineage,seq_length,protein,virus_host_name,geo_location,seq,who_variant,designation,first_designation_date,location,pango_who_annotation
0,YP_009724390.1,surface glycoprotein [Severe acute respiratory...,RefSeq,B,1273,surface glycoprotein,homo sapiens,China,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Wuhan-Hu-1,Index,12/01/2019,China,B (Wuhan-Hu-1)
1,QUR41397.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,Egypt,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Wuhan-Hu-1,Index,12/01/2019,China,B (Wuhan-Hu-1)
2,UIX18801.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,USA: Illinois,MFVFLVLLPLVSSQCVXXTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Wuhan-Hu-1,Index,12/01/2019,China,B (Wuhan-Hu-1)
3,UFJ23379.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,USA: Florida,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Wuhan-Hu-1,Index,12/01/2019,China,B (Wuhan-Hu-1)
4,UGZ97974.1,surface glycoprotein [Severe acute respiratory...,GenBank,B,1273,surface glycoprotein,homo sapiens,USA: California,MFVFLVLLPLVSXXCVNLTTRTXXXPAYTNSFTRGVYYPDKVFRSS...,Wuhan-Hu-1,Index,12/01/2019,China,B (Wuhan-Hu-1)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1521,UNN31516.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1270,surface glycoprotein,homo sapiens,USA: Florida,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Omicron,VOC,11/26/2021,South Africa,BA.1 (Omicron)
1522,WAA85805.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1272,surface glycoprotein,homo sapiens,USA,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Omicron,VOC,11/26/2021,South Africa,BA.1 (Omicron)
1523,UIQ58390.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1270,surface glycoprotein,homo sapiens,USA: South Carolina,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Omicron,VOC,11/26/2021,South Africa,BA.1 (Omicron)
1524,WDU17786.1,surface glycoprotein [Severe acute respiratory...,GenBank,BA.1,1270,surface glycoprotein,homo sapiens,Brazil,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,Omicron,VOC,11/26/2021,South Africa,BA.1 (Omicron)


In [9]:
model_settings = {
    "name": "HAVEN",
    "model_path": os.path.join(os.getcwd(), "..","..", "..", "..", "output/raw/coronaviridae_s_prot_uniref90_embl_vertebrates_t0.01_c8/20240828/host_multi/fine_tuning_hybrid_cls/mlm_tfenc_l6_h8_lr1e-4_uniref90viridae_msl256b512_ae_bn_vs30cls_s64_hybrid_attention_s64_fnn_2l_d1024_lr1e-4_itr4.pth"),
    "segment_len": 256,
    "stride": 64,
    "cls_token": True,
    "n_heads": 8,
    "n_mlp_layers": 2,
    "n_classes": 8,
    "input_dim": 512, # input embedding dimension
    "hidden_dim": 1024,
    "data_parallel": False,
    "pre_train_settings": {
        "n_heads": 8,
        "depth": 6,
        "input_dim": 512, # input embedding dimension
        "hidden_dim": 1024,
        "max_seq_len": 256,
        "vocab_size": constants.VOCAB_SIZE
    }
}


label_settings = {
    "label_col": "virus_host_name",
    "exclude_labels": [ "nan"],
    "label_groupings": {
      "Chicken": [ "gallus gallus" ],
      "Human": [ "homo sapiens" ],
      "Cat": [ "felis catus" ],
      "Pig": [ "sus scrofa" ],
      "Gray wolf": [ "canis lupus" ],
      "Horseshoe bat": ["rhinolophus sp."],
      "Ferret": ["mustela putorius"],
      "Chinese rufous horseshoe bat": ["rhinolophus sinicus"]
    }
}


sequence_settings = {
    "batch_size": 2,
    "id_col": "accession_id",
    "sequence_col": "seq",
    "metadata_cols": ["pangolin_lineage", "geo_location", "seq_length"],
    "truncate": False,
    "split": False,
    "feature_type": "token",
    "max_sequence_length": 256
}

In [10]:
# Choose a sequence from each variant type
selected_input_dfs = [] 
for pango_who_annotation in input_df["pango_who_annotation"].unique():
    selected_input_dfs.append(input_df[input_df["pango_who_annotation]])

SyntaxError: unterminated string literal (detected at line 4) (922957056.py, line 4)

In [ ]:
pre_trained_encoder_model = TransformerEncoder.get_transformer_encoder(model_settings["pre_train_settings"], model_settings["cls_token"])

In [ ]:
model_settings["pre_trained_model"] = pre_trained_encoder_model
haven_model = HAVEN.get_model(model_params=model_settings)

In [ ]:
haven_model.load_state_dict(torch.load(model_settings["model_path"], map_location=nn_utils.get_device()))

In [ ]:
df, index_label_map = utils.transform_labels(input_df, label_settings,
                                                     classification_type="mutli", silent=False)

In [ ]:
sequence_settings["max_sequence_length"]=1281
dataset_loader = dataset_utils.get_dataset_loader(df, sequence_settings, label_col=label_settings["label_col"], include_id_col=True)

In [ ]:
haven_model.eval()

In [ ]:
seq_id, seq, label = next(iter(dataset_loader))

In [ ]:
print(f"seq_id = {seq_id}")
print(f"seq = {seq}, seq_len = {seq.shape}")
print(f"label = {label}")

In [ ]:
x = seq.unfold(dimension=1, size=256, step=64)
x.shape

In [ ]:
output = haven_model(seq)

In [ ]:
output.shape

In [ ]:
output_prob = F.softmax(output, dim=-1)
result_df = pd.DataFrame(output_prob.detach().cpu().numpy())
result_df["id"] = seq_id
result_df["y_true"] = label.detach().cpu().numpy()
idx_label_map={0: 'Cat', 1: 'Chicken', 2: 'Chinese rufous horseshoe bat', 3: 'Ferret', 4: 'Gray wolf', 5: 'Horseshoe bat', 6: 'Human', 7: 'Pig'}
result_df.rename(columns=idx_label_map)

In [ ]:
seq_len = seq.shape[1]
pos_mapping_range = {}
pos_mapping = {}
j = 0
for i in range(0, seq_len + 1, 64):
    start = i
    end = i + 256
    if end >= seq_len:
        break
    pos_mapping[j] = f"{j}: {start + 1}-{end}"
    pos_mapping_range[j] = [start, end]
    j += 1
    
pos_mapping

In [ ]:
pos_mapping_range

In [ ]:
pos_n_segments_map = {}
for i in range(1273):
    for _, segment in pos_mapping_range.items():
        if i >= segment[0] and i < segment[1]:
            if i in pos_n_segments_map:
                pos_n_segments_map[i] += 1
            else:
                pos_n_segments_map[i] = 1

In [ ]:
idx=0
inter_segment_attn = haven_model.self_attn.self_attn[idx, :, :, :]
inter_segment_attn.shape

In [ ]:
# TODO: Check whether normalization is applicable here
fig, axs = plt.subplots(4, 2, figsize=(16, 32), sharex=False, sharey=False)
axs = axs.flatten()
for i in range(8):
    ax = axs[i]
    max_values = torch.max(inter_segment_attn[i]) # max value across the entire head
    plot_df = pd.DataFrame((inter_segment_attn[i]/max_values).detach().cpu().numpy()).rename(columns=pos_mapping, index=pos_mapping)
    #plot_df = pd.DataFrame(attn[i].detach().cpu().numpy()).rename(columns=pos_mapping, index=pos_mapping)
    sns.heatmap(plot_df, cmap="crest", ax=ax,)# cbar=i%2==1)
    ax.set_title(f"Head {i}")
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.25)
plt.savefig(os.path.join(os.getcwd(), "..","..", "..", "..", "output/visualization/haven/interpretability_wiv04_segment_attn_norm_all_heads.pdf"), dpi=1024, bbox_inches="tight")

In [ ]:
mean_inter_seg_attn = inter_segment_attn.mean(dim=0)
plot_df = pd.DataFrame(mean_inter_seg_attn.detach().cpu().numpy()).rename(columns=pos_mapping, index=pos_mapping)
ax = sns.heatmap(plot_df, cmap="crest")
ax.set_title(f"Segment Attention (Mean across all heads)")
plt.savefig(os.path.join(os.getcwd(), "..","..", "..", "..", "output/visualization/haven/interpretability_wiv04_segment_attn_heads_mean.pdf"), dpi=1024, bbox_inches="tight")

In [ ]:
mean_inter_seg_attn = inter_segment_attn.mean(dim=0)
plot_df = pd.DataFrame((mean_inter_seg_attn/(mean_inter_seg_attn.max())).detach().cpu().numpy()).rename(columns=pos_mapping, index=pos_mapping)
ax = sns.heatmap(plot_df, cmap="crest")
ax.set_title(f"Segment Attention (Mean across all heads)")
plt.savefig(os.path.join(os.getcwd(), "..","..", "..", "..", "output/visualization/haven/interpretability_wiv04_segment_attn_heads_mean_norm.pdf"), dpi=1024, bbox_inches="tight")

# Attention for every layer aggregated across all segments and all heads

In [ ]:
def compute_segment_embedding(X):
    batch_size = X.shape[0]
    X = X.unfold(dimension=1, size=haven_model.segment_len, step=haven_model.stride)
    X = X.contiguous().view(-1, haven_model.segment_len)
    cls_tokens = torch.full(size=(X.shape[0], 1), fill_value=constants.CLS_TOKEN_VAL,
                                    device=nn_utils.get_device())
    
    X = torch.cat([cls_tokens, X], dim=1)
    X = haven_model.pre_trained_model(X, mask=None)    
    X = X.view(batch_size, -1, haven_model.segment_len + 1, haven_model.input_dim)
    X = X[:, :, 0, :]
    X = haven_model.self_attn(X, X, X)
    return haven_model.feed_forward(X).mean(dim=1)

In [ ]:
pos_attn_values = []
compute_segment_embedding(seq[idx].unsqueeze(0))
for enc_layer in range(6):
    for i in range(1273):
        pos = i
        pos_attn_values_in_layer = []
        # traverse the pos_segment map
        for segment_idx, segment in pos_mapping_range.items():
            if i >= segment[0] and i < segment[1]:
                pos_idx_in_seg = i - segment[0]
                mean_inter_segment_attn = inter_segment_attn.mean(dim=0).mean(dim=0)
                segment_attn_val = (mean_inter_segment_attn/(mean_inter_segment_attn.max()))[segment_idx].item()
                
                intra_segment_attn=haven_model.pre_trained_model.encoder.layers[enc_layer].self_attn.self_attn[segment_idx]
                mean_intra_segment_attn = intra_segment_attn.mean(dim=0).mean(dim=0)
                pos_attn_val = (mean_intra_segment_attn/(mean_intra_segment_attn.max()))[pos_idx_in_seg+1].item() # +1 to account for CLS token
                pos_attn_values_in_layer.append(segment_attn_val*pos_attn_val)
        pos_attn_values.append({
            "pos": pos+1,
            "enc_layer": enc_layer,
            "absolute_pos_attention": statistics.mean(pos_attn_values_in_layer)
        })
cumulative_pos_attn_values_df = pd.DataFrame(pos_attn_values)
cumulative_pos_attn_values_df["pos"] = cumulative_pos_attn_values_df["pos"].apply(str)

In [ ]:
k = 100
fig, axs = plt.subplots(6, 1, figsize=(20, 24), sharex=False, sharey=True)
axs = axs.flatten()
for enc_layer in range(6):
    ax = axs[enc_layer]
    enc_df = cumulative_pos_attn_values_df[cumulative_pos_attn_values_df["enc_layer"] == enc_layer].sort_values("absolute_pos_attention", ascending=False)
    sns.lineplot(data = enc_df[:k], x="pos", y="absolute_pos_attention", ax=ax)
    ax.set_title(f"Encoder Layer = {enc_layer} (Top {k} positions)")
    ax.tick_params(labelrotation=90)
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.55)

# Attention for every position within every segment in every layer aggregated across all heads (not accounting for segment attention)

In [ ]:
compute_segment_embedding(seq[idx].unsqueeze(0))
fig, axs = plt.subplots(17, 6, figsize=(24, 48), sharex=False, sharey=False)
axs = axs.flatten()

i = 0
for segment_idx, segment in pos_mapping_range.items():
    position_mapping = {}
    j = 1
    for pos in range(segment[0], segment[1]):
        position_mapping[j] = pos + 1
        j +=1
    for enc_layer in range(6):
        ax = axs[i]
        intra_segment_attn=haven_model.pre_trained_model.encoder.layers[enc_layer].self_attn.self_attn[segment_idx]
        mean_intra_segment_attn = intra_segment_attn.mean(dim=0)
        mean_intra_segment_attn_norm = mean_intra_segment_attn/(mean_intra_segment_attn.max())
        plot_df = pd.DataFrame(mean_intra_segment_attn_norm.detach().cpu().numpy()).rename(columns=position_mapping, index=position_mapping)
        sns.heatmap(plot_df, cmap="crest", ax=ax, cbar=(i%6 == 5))
        ax.set_title(f"Segment {segment}, Encoder Layer {enc_layer}")
        i += 1
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.55)

In [ ]:
fig, axs = plt.subplots(17, 6, figsize=(48, 48), sharex=False, sharey=False)
axs = axs.flatten()

k = 50
i = 0
pos_count = {}
for segment_idx, segment in pos_mapping_range.items():
    position_mapping = {}
    j = 1
    for pos in range(segment[0], segment[1]):
        position_mapping[j] = pos + 1
        j +=1
    for enc_layer in range(6):
        ax = axs[i]
        intra_segment_attn=haven_model.pre_trained_model.encoder.layers[enc_layer].self_attn.self_attn[segment_idx]
        mean_intra_segment_attn = intra_segment_attn.mean(dim=0).mean(dim=0)
        mean_intra_segment_attn_norm = mean_intra_segment_attn/(mean_intra_segment_attn.max())
        plot_df = pd.DataFrame(mean_intra_segment_attn_norm.detach().cpu().numpy())[1:] # :1 is to account for CLS token
        plot_df = plot_df.rename(index=position_mapping).reset_index().rename(columns={"index": "pos", 0: "agg_pos_attention"}).sort_values(by="agg_pos_attention", ascending=False)
        plot_df["pos"] = plot_df["pos"].apply(str)
        sns.lineplot(data=plot_df[:k], x="pos", y="agg_pos_attention", ax=ax)
        ax.set_title(f"Segment {segment}, Encoder Layer {enc_layer}")
        ax.tick_params(labelrotation=90)
        i += 1
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.55)

In [ ]:
k = 10
top_k_pos_count = {}
seg_enc_dfs = []
for segment_idx, segment in pos_mapping_range.items():
    position_mapping = {}
    j = 1
    for pos in range(segment[0], segment[1]):
        position_mapping[j] = pos + 1
        j +=1
    for enc_layer in range(6):
        intra_segment_attn=haven_model.pre_trained_model.encoder.layers[enc_layer].self_attn.self_attn[segment_idx]
        mean_intra_segment_attn = intra_segment_attn.mean(dim=0).mean(dim=0)
        mean_intra_segment_attn_norm = mean_intra_segment_attn/(mean_intra_segment_attn.max())
        seg_enc_df = pd.DataFrame(mean_intra_segment_attn_norm.detach().cpu().numpy())[1:] # :1 is to account for CLS token
        seg_enc_df = seg_enc_df.rename(index=position_mapping).reset_index().rename(columns={"index": "pos", 0: "agg_pos_attention"}).sort_values(by="agg_pos_attention", ascending=False)
        # add rank column
        seg_enc_df = seg_enc_df.reset_index(drop=True).reset_index().rename(columns={"index": "rank"})
        seg_enc_df["rank"] += 1
        seg_enc_df["enc_layer"] = enc_layer
        seg_enc_df["segment"] = segment_idx
        seg_enc_dfs.append(seg_enc_df)
        top_k_positions = seg_enc_df["pos"].values[:k]
        for x in top_k_positions:
            if x in top_k_pos_count:
                top_k_pos_count[x] += 1
            else:
                top_k_pos_count[x] = 1
        seg_enc_df["pos"] = seg_enc_df["pos"].apply(str)

for key, val in top_k_pos_count.items():
    top_k_pos_count[key] = val/pos_n_segments_map[key - 1]
top_k_pos_count_df = pd.DataFrame.from_dict(top_k_pos_count, orient="index", columns=["top_k_pos_count_corr"]).reset_index().rename(columns={"index": "pos"})#, "top_k_pos_count": "top_k_pos_count_corr"})
#top_k_pos_count_df["top_k_pos_count_corr"] = top_k_pos_count["pos", "top_k_pos_count"].apply(lambda x: x[1]/pos_n_segments_map[x[0]-1])
seg_enc_df = pd.concat(seg_enc_dfs)
seg_enc_df["imp_score"] = 1/seg_enc_df["rank"]

In [ ]:
top_k_pos_count_df.sort_values("top_k_pos_count_corr", ascending=False).head(25)

In [ ]:
seg_enc_df

In [ ]:
fig, axs = plt.subplots(10, 1, figsize=(24, 24), sharex=False, sharey=True)
axs = axs.flatten()
i = 0
bin_width = 128
for start in range(1, 1274, bin_width):
    ax = axs[i]
    i += 1
    end = start + bin_width
    plot_df = seg_enc_df[(seg_enc_df["pos"].astype(int) >= start) & (seg_enc_df["pos"].astype(int) < end)]
    plot_df["pos"] = plot_df["pos"].apply(int)
    sns.stripplot(plot_df, x="pos", y="imp_score", ax=ax)
    ax.tick_params(labelrotation=90)
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.1)

In [ ]:
fig, axs = plt.subplots(17, 1, figsize=(24, 48), sharex=False, sharey=False)
axs = axs.flatten()

i = 0
for segment_idx, segment in pos_mapping_range.items():
    ax = axs[i]
    i += 1
    positions = {}
    j = 1
    for pos in range(segment[0], segment[1]):
        positions[j] = pos + 1
        j +=1
    segment_attn_values=[]
    for enc_layer in range(6):
        segment_attn_matrix=haven_model.pre_trained_model.encoder.layers[enc_layer].self_attn.self_attn[segment_idx]
        pos_attn_agg = normalize(segment_attn_matrix.mean(dim=0).mean(dim=0), p=2, dim=0)
        segment_attn_values.append(pd.DataFrame(pos_attn_agg.detach().cpu().numpy()).rename(index=positions).transpose().set_index(pd.Series([enc_layer+1])))
    sns.heatmap(pd.concat(segment_attn_values), cmap="crest", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"Segment {segment}")
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.55)

In [ ]:
fig, axs = plt.subplots(17, 1, figsize=(24, 48), sharex=False, sharey=False)
axs = axs.flatten()

i = 0
for segment_idx, segment in pos_mapping_range.items():
    ax = axs[i]
    i += 1
    positions = {}
    j = 1
    for pos in range(segment[0], segment[1]):
        positions[j] = pos + 1
        j +=1
    segment_attn_values=[]
    for enc_layer in range(6):
        segment_attn_matrix=haven_model.pre_trained_model.encoder.layers[enc_layer].self_attn.self_attn[segment_idx]
        pos_attn_agg = normalize(segment_attn_matrix[-1].mean(dim=0), p=2, dim=0)
        segment_attn_values.append(pd.DataFrame(pos_attn_agg.detach().cpu().numpy()).rename(index=positions).transpose().set_index(pd.Series([enc_layer+1])))
    sns.heatmap(pd.concat(segment_attn_values), cmap="crest", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"Segment {segment}")
plt.subplots_adjust(wspace=0.2)
plt.subplots_adjust(hspace=0.55)